# C1 — OU Half-Life per Roll Day

In [ ]:
import sys, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, '.')
sys.path.insert(0, '../notebooks')
from config import (RESULTS_DIR, SEAGATE_DIR, FIGS_DIR, WINDOWS_META,
                    BASELINE_STATS, UPDATED_STATS, WIN_COLORS,
                    TICK, MULT, save_fig)
Path('figures').mkdir(exist_ok=True)


In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('hl_mod', '../notebooks/11_half_life_bars.py')
hl_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(hl_mod)
hl_series_minutes = hl_mod.hl_series_minutes


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for wk, wm in WINDOWS_META.items():
    rk = wm['result_key']
    ts = pd.read_parquet(RESULTS_DIR / rk / 'none' / 'timeseries.parquet')
    ts.index = pd.to_datetime(ts.index)
    ts['date'] = ts.index.date.astype(str)
    day_hl = []
    for d in wm['days']:
        sub = ts[ts['date'] == d]['dev'].dropna().values
        if len(sub) < 10:
            day_hl.append(np.nan); continue
        try:
            hl = hl_series_minutes(sub, bar_res=1, n_bars=len(sub), bar_secs=1)
            day_hl.append(np.nanmedian(hl))
        except Exception:
            day_hl.append(np.nan)
    ax.plot(range(len(wm['days'])), day_hl, marker='o', label=wk, color=WIN_COLORS[wk])
ax.set_xticks(range(7))
ax.set_xticklabels(['D1','D2','D3','D4','D5','D6','D7'])
ax.set_xlabel('Roll Day')
ax.set_ylabel('Median OU Half-Life (minutes)')
ax.set_title('OU Half-Life per Roll Day — All Windows')
ax.legend()
save_fig(fig, 'C1_ou_halflife_per_rollday.png')
